# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Rank content by refresh priority score based on:
High impressions but low CTR → underperforming
Declining trend → losing relevance
Poor ranking position → optimization opportunity

Each row gets:

Action (what to do)
Reason code (why it was chosen)

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load dataset
df = pd.read_csv("https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv")

# Scoring function
df["score"] = (
    (df["impressions_90d"] * (1 - df["ctr"])) +   # high impressions + low CTR
    ((20 - df["avg_position"]).clip(lower=0) * 10) +  # worse rank = higher score
    (-df["trend_pct"])  # negative trend = higher priority
)

# Actions
def get_action(row):
    if row["ctr"] < 0.02 and row["impressions_90d"] > 1000:
        return "Improve title/meta (CTR fix)"
    elif row["avg_position"] > 10:
        return "Update content for SEO (ranking fix)"
    elif row["trend_pct"] < 0:
        return "Refresh content (declining traffic)"
    else:
        return "No action"

# Reason codes
def get_reason(row):
    if row["ctr"] < 0.02:
        return "Low CTR"
    elif row["avg_position"] > 10:
        return "Poor ranking"
    elif row["trend_pct"] < 0:
        return "Traffic declining"
    else:
        return "Healthy"

df["action"] = df.apply(get_action, axis=1)
df["reason"] = df.apply(get_reason, axis=1)

# Ranked queue
queue = df.sort_values(by="score", ascending=False)

queue[["content_id", "score", "action", "reason"]].head()

,content_id,score,action,reason
19636,content_2cb567c3c89b,447931.20,Update content for SEO (ranking fix),Poor ranking
6653,content_5fe46e04994d,445437.70,Refresh content (declining traffic),Traffic declining
26844,content_8c19996aa890,433083.70,Refresh content (declining traffic),Traffic declining
17812,content_aaef01a50def,387981.75,Refresh content (declining traffic),Traffic declining
29400,content_2dba2b1f9536,350311.46,Update content for SEO (ranking fix),Poor ranking


The queue prioritizes:

Pages wasting impressions (low CTR)
Pages losing traffic (negative trend)
Pages ranking poorly but close to page 1

This creates a clear, trustable order of work.

In [2]:
# Top 20 priority items
top_queue = queue[[
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "trend_pct",
    "action",
    "reason",
    "score"
]].head(20)

print(top_queue)

                 content_id  impressions_90d   ctr  avg_position  trend_pct  \
19636  content_2cb567c3c89b           497727  0.10          22.2       23.1   
6653   content_5fe46e04994d           517715  0.14           4.2      -44.8   
26844  content_8c19996aa890           509252  0.15           2.5      -44.5   
17812  content_aaef01a50def           517109  0.25           5.4       -4.0   
29400  content_2dba2b1f9536           443434  0.21          27.9        1.4   
29879  content_1a9e894be2e2           416180  0.23           4.0      -27.0   
3394   content_36ff89c8214e           295097  0.05           7.3        0.5   
21819  content_4c36c775b818           463103  0.41           2.3      -33.2   
18870  content_db5989a78dd3           345111  0.21           5.4      556.2   
26798  content_b28d1efd668f           286608  0.06          26.2      -17.2   
7678   content_8451fc6f034d           272144  0.03           2.3       73.1   
26531  content_cb112fce36be           309910  0.16  

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use:

SEO/content teams use this to decide what to update first
Helps prioritize high-impact content refreshes

Limits:

Based only on historical data (90d + trends)
Does NOT include:
seasonality
competitor changes
content quality manually judged
Assumes CTR + rank = opportunity (not always true)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human must check before acting:

Content accuracy (facts outdated?)
Search intent match
Brand tone & quality
Duplicate or cannibalized pages

Never automate:

Deleting content
Major rewrites without review
Changing intent/topic blindly
Publishing AI content without validation

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Retrain or update model when:

CTR distribution shifts significantly
Avg position improves but traffic drops
Trend_pct becomes unstable
New content types appear
Google algorithm update detected

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Monitoring metrics
print("Avg CTR:", df["ctr"].mean())
print("Avg Position:", df["avg_position"].mean())
print("Avg Trend:", df["trend_pct"].mean())

# Drift check example
ctr_std = df["ctr"].std()
trend_std = df["trend_pct"].std()

print("CTR std:", ctr_std)
print("Trend std:", trend_std)

Avg CTR: 0.5107333333333334
Avg Position: 16.342380000000002
Avg Trend: -4.785968735908612
CTR std: 3.2791619679929336
Trend std: 473.8617795137961


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

I export:

Ranked queue
Top recommendations
Any charts/data used in paper

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Create folder
os.makedirs("work/outputs", exist_ok=True)

# Save full queue
queue.to_csv("work/outputs/full_queue.csv", index=False)

# Save top 20
top_queue.to_csv("work/outputs/top_queue.csv", index=False)

# Optional: summary stats
df.describe().to_csv("work/outputs/summary_stats.csv")

print("Exports saved to work/outputs/")

Exports saved to work/outputs/


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.